<a href="https://colab.research.google.com/github/Chaki34/python-basics/blob/main/MultiProcessing%20tutorial%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# without maltiprocessing

import time


def testing():
    total = 0
    for i in range(100_000_000):
        total += i


start = time.time()

testing()

print("Main process: Doing another task...")
for i in range(5):
    print("Main task:", i)


end = time.time()

print("Execution is completed")
print("Execution time:", end - start, "seconds")

Main process: Doing another task...
Main task: 0
Main task: 1
Main task: 2
Main task: 3
Main task: 4
Execution is completed
Execution time: 5.79067063331604 seconds


In [14]:
# Multiprocessing
import multiprocessing as mp
import time


# Function that will run inside a separate process
def testing():
    total = 0
    for i in range(100_000_000):
        total += i


# The __main__ check is important in multiprocessing.
# It prevents the process-creation code from running again
# when a new process imports this module.
if __name__ == '__main__':

    # Create a new process and assign the testing function to it
    m = mp.Process(target=testing)

    # Record the starting time
    start = time.time()

    # Start the process
    m.start()

    # another task
    print("Main process: Doing another task...")
    for i in range(5):
        print("Main task:", i)


    # Wait until the process finishes
    m.join()

    # Record the ending time
    end = time.time()

    print("Process execution is completed")
    print("Execution time:", end - start, "seconds")

Main process: Doing another task...
Main task: 0
Main task: 1
Main task: 2
Main task: 3
Main task: 4
Process execution is completed
Execution time: 5.500114679336548 seconds


In [17]:
import multiprocessing as mp
import time


# Process Pool:
# A Pool creates multiple worker processes and distributes
# independent tasks among those workers.
#
# Important concepts:
# 1. Pool(processes=n) -> creates n worker processes
# 2. map() -> distributes tasks and returns results in order
# 3. Worker process -> executes the given function
# 4. Parallel execution -> multiple tasks can run at the same time
# 5. Pool automatically manages the worker processes


def square(n):
    return n ** 20


if __name__ == '__main__':

    start = time.time()

    with mp.Pool(processes=4) as pool:
        output = pool.map(square, [1, 2, 3, 4, 5, 6, 7, 8, 9])

    end_time = time.time()

    print("Execution time:", end_time - start, "seconds")
    print(output)

Execution time: 0.0019154548645019531 seconds
[1, 1048576, 3486784401, 1099511627776, 95367431640625, 3656158440062976, 79792266297612001, 1152921504606846976, 12157665459056928801]


In [27]:
# Queue
import multiprocessing as mp
import random

# Queue is used for communication between processes.
# It follows FIFO (First In, First Out).
# Producer puts data into the Queue.
def producer(q):
    for i in range(10):
         q.put(chr(random.randint(97, 122)))

    # None is used as a signal to tell the consumer
    # that there is no more data to process.
    q.put(None)


# Consumer gets data from the Queue.
def consumer(q):
    while True:
        item = q.get()

        # Stop when the producer sends None.
        if item is None:
            break

        print("Consumed:", item)


if __name__ == '__main__':

    # Create a multiprocessing Queue.
    q = mp.Queue()

    # Create producer and consumer processes.
    start = time.time()
    p1 = mp.Process(target=producer, args=(q,))
    p2 = mp.Process(target=consumer, args=(q,))

    # Start both processes.
    p1.start()
    p2.start()

    # Wait for both processes to finish.
    p1.join()
    p2.join()
    end = time.time()

    print("Execution time:", end - start, "seconds")

    print("Processing completed")

Consumed: h
Consumed: x
Consumed: w
Consumed: n
Consumed: o
Consumed: b
Consumed: w
Consumed: o
Consumed: m
Consumed: h
Execution time: 0.11227798461914062 seconds
Processing completed


In [31]:
# Array
import multiprocessing as mp
import time


# mp.Array creates shared memory that can be accessed
# by multiple processes.
# Here, each process squares one element of the array.
def square(index, value):
    value[index] = value[index] ** 2


if __name__ == '__main__':

    # Create a shared integer array.
    # 'i' means signed integer.
    arr = mp.Array('i', [1, 2, 3, 4, 5, 6, 7, 8, 9])

    # Store all process objects.
    processes = []

    start = time.time()

    # Create one process for each array element.
    for i in range(9):
        p = mp.Process(target=square, args=(i, arr))
        processes.append(p)
        p.start()

    # Wait for all processes to finish.
    for p in processes:
        p.join()

    end = time.time()

    print("Array:", list(arr))
    print("Execution Time:", end - start, "seconds")

[1, 4, 9, 16, 25, 36, 49, 64, 81]
Execution Time :-0.10775876045227051


In [36]:
# Pipe
import multiprocessing as mp
import random


# Pipe is used for communication between two processes.
# It provides two connection objects:
# one for sending data and one for receiving data.
#
# Unlike Queue, Pipe is mainly designed for
# communication between two processes.
def sender(conn):

    sentences = [
        "Python is easy to learn",
        "Multiprocessing improves performance",
        "Queue is used for process communication",
        "Producer sends data to consumer",
        "Consumer processes the received data",
        "Python supports parallel programming",
        "Processes can run independently",
        "A queue follows FIFO order"
    ]

    # Send random sentences through the Pipe.
    for i in range(10):
        conn.send(random.choice(sentences))

    # Send None as a signal that no more data is available.
    conn.send(None)

    # Close the connection after sending all data.
    conn.close()


def receiver(conn):

    while True:
        msg = conn.recv()

        # Stop receiving when None is received.
        if msg is None:
            break

        print("Received:", msg)

    # Close the connection.
    conn.close()


if __name__ == '__main__':

    # Create two connection objects.
    # The first is used for sending and the second for receiving.
    sender_conn, receiver_conn = mp.Pipe()

    # Create sender and receiver processes.
    p1 = mp.Process(target=sender, args=(sender_conn,))
    p2 = mp.Process(target=receiver, args=(receiver_conn,))

    p1.start()
    p2.start()

    # Parent process does not need these connections.
    sender_conn.close()
    receiver_conn.close()

    # Wait for both processes to finish.
    p1.join()
    p2.join()

    print("Communication completed")

Received: Consumer processes the received data
Received: Consumer processes the received data
Received: A queue follows FIFO order
Received: A queue follows FIFO order
Received: Python supports parallel programming
Received: Consumer processes the received data
Received: Multiprocessing improves performance
Received: Python is easy to learn
Received: Processes can run independently
Received: Python is easy to learn
Communication completed
